![Diagram](./images/Metadata_Filter.png)

In [39]:
import os
import json
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
import warnings
warnings.filterwarnings("ignore")

In [40]:
load_dotenv()

True

# Enhancing RAG with Metadata

In the first notebook we built a basic RAG pipeline over the **ShopEasy** customer-support knowledge base. Plain semantic search worked for natural language, but it returned noise: ask about a shipping problem and you'd also get returns and refund docs because the language overlaps.

Now we'll make retrieval smarter using **metadata**.

__What is metadata?__ It's data about your data. Every document in our knowledge base already carries a structured `metadata` object — `doc_type`, `product_area`, `priority`, `platform`, `customer_tier`, and `status`. Pinecone stores this metadata *alongside each vector*, so we can **filter** the search space before running a semantic query — results come back pre-scoped to exactly the right context.

> **We reuse the vectors from Notebook 1.** The basic-RAG pipeline already indexed these documents (with their metadata) into the `shopeasy-basic-rag` namespace. Metadata filtering needs no new vectors — it's just a query-time filter on what's already there. So this notebook **connects to that same namespace** instead of re-indexing. Make sure you've run Notebook 1 first.

We'll cover two ways to filter:

1. **Manual filters** — the agent specifies the filter directly (great for operational queries like "show me all active P1 order bugs").
2. **LLM-classified filters** — an LLM reads the customer ticket and auto-selects the right filter (product area, doc type, customer tier).

# Connect to the existing index

We connect to the **same** Pinecone index and the **same** `shopeasy-basic-rag` namespace that Notebook 1 populated, using the identical embedding model (`text-embedding-3-small` at 512 dims). No loading, chunking, or indexing is needed here — those vectors and their metadata already live in Pinecone.

This expects `PINECONE_API_KEY`, `PINECONE_INDEX_NAME`, and `OPENAI_API_KEY` in your environment.

In [41]:
#define the embeddings model (must match Notebook 1: 512 dims)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small", dimensions=512)  # 512-dim vectors

#connect to the existing Pinecone index + the namespace Notebook 1 already populated
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
index = pc.Index(os.environ["PINECONE_INDEX_NAME"])

vector_store = PineconeVectorStore(
    index=index,
    embedding=embeddings,
    namespace="shopeasy-basic-rag",
)

#sanity check: how many vectors are in our namespace?
stats = index.describe_index_stats()
print(stats["namespaces"].get("shopeasy-basic-rag"))

{'vector_count': 193}


### What can we filter on?

The chunks in the index carry these metadata fields. To see the distinct values available for each one (the "facets" we can slice by), we read them straight from the source JSON — the same data that was indexed.

![Diagram](./images/08_metedata_facets.png)

In [42]:
# Load the source knowledge base just to inspect the available metadata values.
with open("shopeasy_knowledge_base.json") as f:
    kb = json.load(f)

filterable_fields = ["doc_type", "product_area", "priority", "platform", "customer_tier", "status"]

for field in filterable_fields:
    values = sorted({entry["metadata"].get(field) for entry in kb})
    print(f"{field:15s}: {values}")

doc_type       : ['bug_report', 'faq', 'past_ticket', 'product_doc', 'runbook']
product_area   : ['account', 'orders', 'payments', 'returns', 'shipping']
priority       : ['P0', 'P1', 'P2', 'P3']
platform       : ['all', 'mobile', 'web']
customer_tier  : ['all', 'plus', 'regular']
status         : ['active', 'resolved']


# Retrieval & Generation (baseline, no filter)

First, let's set up our LLM and prompt template and run a query **without** any metadata filter — so we can see the noise that filtering will later remove.

![Diagram](./images/09_baseline_retrieval.png)

In [43]:
#configure the llm
llm = ChatOpenAI(model="gpt-4.1-mini")

#set the prompt template
template = """You are a customer-support assistant for ShopEasy, an e-commerce platform.
Use the following pieces of retrieved internal knowledge-base context to help resolve the customer's issue.
If the context doesn't contain the answer, say you don't have that information rather than guessing.
Be concise and practical: state the likely cause and the next step the agent should take.

Context:
{context}

Customer issue: {question}

Support guidance:"""

rag_prompt_template = PromptTemplate.from_template(template)

In [44]:
# A natural "is this a known bug?" question. The customer wants BUG REPORTS,
# but plain semantic search has no notion of doc types -- it just matches words.
user_question = "Are there any known bugs with the checkout process?"

# Retrieve WITHOUT a filter and print the metadata header so we can SEE the noise.
retriever = vector_store.as_retriever(search_kwargs={"k": 5}, search_type="similarity")
retrieved_docs = retriever.invoke(user_question)

for doc in retrieved_docs:
    m = doc.metadata
    print(f"[{m['doc_type']} / {m['product_area']}] {m['title']}")

# Notice the problem: the top hits are a PAST TICKET (twice), a runbook and a
# product doc -- not a single bug_report. The actual bug report we want
# ("Mobile app crashes at checkout on Android 14") didn't even make the top 5.
# The words "checkout"/"crash" match many doc types, so semantic search alone
# can't give us "only known bugs."

[past_ticket / payments] Mobile app checkout crashes on Android — CD-8456
[past_ticket / payments] Mobile app checkout crashes on Android — CD-8456
[runbook / orders] Subscribe & Save Issues
[runbook / payments] Troubleshooting Promo Code Failures (PROMO_INVALID_100)
[product_doc / payments] Payment Methods and Billing


# Retrieval with Metadata Filtering

This is where the magic happens! Because the metadata is indexed alongside each vector, we can use it to make retrieval far more precise.

### Manual Metadata Filtering

The fix is to filter on metadata *before* the semantic search runs. Pinecone supports operators like `$eq`, `$ne`, `$in`, `$gt`, ...

For our "known checkout bugs" question we scope to exactly what we want: `doc_type = bug_report` and `product_area = payments`. The only results that come back are actual payment/checkout bug reports -- the noise from the baseline run is gone.

![Diagram](./images/10_manual_filtering.png)

In [45]:
# Scope to checkout/payments bug reports only.
bug_filter = {
    "doc_type": {"$eq": "bug_report"},
    "product_area": {"$eq": "payments"},
}

retriever = vector_store.as_retriever(
    search_kwargs={"k": 5, "filter": bug_filter}, search_type="similarity"
)
retrieved_docs = retriever.invoke(user_question)

# Every result is now an actual bug report in the payments area.
for doc in retrieved_docs:
    m = doc.metadata
    print(f"[{m['doc_type']} / {m['product_area']}] {m['title']}")
    print(doc.page_content)
    print("-" * 100)

[bug_report / payments] Mobile app crashes at checkout on Android 14
Impact: Approximately 8% of Android checkout attempts are failing. Affected users can still complete orders via the mobile website (m.shopeasy.com) or by disabling Google Pay in the app and using manual card entry.

Workaround: Advise customers to either (a) use the mobile website m.shopeasy.com, or (b) disable Google Pay in the app: Settings > Payment > Google Pay > Off, then enter card details manually at checkout.
----------------------------------------------------------------------------------------------------
[bug_report / payments] Mobile app crashes at checkout on Android 14
Bug CD-8456 — Mobile app crashes at checkout on Android 14 with December 2025 security patch

Severity: P2
Status: Active
Affected Versions: App v4.6.0 – v4.7.1
Platform: Android (Samsung Galaxy S23/S24, Pixel 8/9)
----------------------------------------------------------------------------------------------------
[bug_report / payments] 

### Auto-classifying the filter with an LLM

Manually creating filters is great, but what if the LLM could build the filter for us based on the customer's ticket?

We'll use an LLM with **structured output** to read the ticket and classify it into the relevant `product_area`, and — when the ticket clearly calls for it — a `doc_type` and `customer_tier`. We then assemble those into a Pinecone filter automatically.

_ps: You can also use LangChain's Self Query Retriever for this._

![Diagram](./images/11_llm_classified_filters.png)

In [46]:
from typing import Optional
from pydantic import BaseModel, Field

# Define the output schema using a Pydantic model.
# product_area is always required; doc_type / customer_tier are optional and only
# set when the ticket clearly points at them.
class TicketFilter(BaseModel):
    """Structured metadata filter inferred from a customer support ticket."""
    product_area: str = Field(
        description="One of: payments, returns, shipping, orders, account"
    )
    doc_type: Optional[str] = Field(
        default=None,
        description="Optionally one of: runbook, past_ticket, product_doc, bug_report, faq. "
                    "Set only if the ticket clearly targets a doc type (e.g. asking about a known bug -> bug_report).",
    )
    customer_tier: Optional[str] = Field(
        default=None,
        description="Optionally one of: plus, regular. Set only if the customer's tier is explicitly mentioned.",
    )

classifier_llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)
structured_llm = classifier_llm.with_structured_output(TicketFilter)

classify_template = """You classify ShopEasy customer support tickets so we can filter the knowledge base.
Read the ticket and return the metadata that best scopes the search.

Ticket: {question}
"""
classify_prompt_template = PromptTemplate.from_template(classify_template)

We convert the classified fields into a Pinecone filter, skipping any field the LLM left empty.

In [47]:
def build_pinecone_filter(ticket_filter: TicketFilter) -> dict:
    """Turn the classified fields into a Pinecone metadata filter, skipping empty ones.

    customer_tier is special: a value of "all" in the data means "applies to every
    tier". So for a 'plus' (or 'regular') member we match BOTH that tier AND "all"
    via $in -- otherwise an $eq filter would silently drop the general docs that
    also apply to this customer (and can leave us with zero results).
    """
    pinecone_filter = {}
    for field, value in ticket_filter.model_dump().items():
        if not value:
            continue
        if field == "customer_tier":
            pinecone_filter[field] = {"$in": [value, "all"]}
        else:
            pinecone_filter[field] = {"$eq": value}
    return pinecone_filter


# A clear tier-specific ticket: the customer states they are a Plus member and asks
# about returns. The classifier should set product_area=returns and customer_tier=plus.
user_question = "I'm a ShopEasy Plus member -- how many days do I have to return an opened pair of headphones?"

prompt = classify_prompt_template.invoke({"question": user_question})
ticket_filter = structured_llm.invoke(prompt)
print("Classified:", ticket_filter)

pinecone_filter = build_pinecone_filter(ticket_filter)
print("Pinecone filter:", pinecone_filter)

Classified: product_area='returns' doc_type='faq' customer_tier='plus'
Pinecone filter: {'product_area': {'$eq': 'returns'}, 'doc_type': {'$eq': 'faq'}, 'customer_tier': {'$in': ['plus', 'all']}}


In [48]:
def generate_filtered_answer(user_question, pinecone_filter):
    #retrieve the relevant docs, scoped by the metadata filter
    retriever = vector_store.as_retriever(
        search_kwargs={"k": 5, "filter": pinecone_filter}, search_type="similarity"
    )
    retrieved_docs = retriever.invoke(user_question)

    #generate
    docs_content = "\n\n".join(doc.page_content for doc in retrieved_docs)
    prompt = rag_prompt_template.invoke({"question": user_question, "context": docs_content})
    response = llm.invoke(prompt)

    return retrieved_docs, response.content


retrieved_docs, answer = generate_filtered_answer(user_question, pinecone_filter)

In [51]:
# Every returns doc in this KB is tier="all" -- there is no Plus-specific returns doc.
# A naive customer_tier == "plus" ($eq) filter would return NOTHING here; the
# $in: ["plus", "all"] logic keeps the general Returns runbook, which actually holds
# the answer (standard 30 days, electronics 15 days, Plus members 60 days).
for doc in retrieved_docs:
    m = doc.metadata
    print(f"[{m['doc_type']} / {m['product_area']} / tier={m['customer_tier']}] {m['title']}")
    print(doc.page_content)
    print("-" * 100)

[faq / returns / tier=all] What is ShopEasy's return policy?
FAQ: What is ShopEasy's return policy?

Most items can be returned within 30 days of delivery for a full refund. Electronics have a shorter 15-day window and must be unopened for change-of-mind returns (defective items can be returned opened). Final Sale items are non-returnable.

ShopEasy Plus members get an extended 60-day return window and free return shipping. Standard members pay $7.95 for return shipping unless the item is defective or was the wrong item.
----------------------------------------------------------------------------------------------------
[faq / returns / tier=all] What is ShopEasy's return policy?
To start a return, go to "My Orders" in your account, find the order, and click "Return Item." Select the correct reason code — this determines whether return shipping is free. Refunds are processed within 3-5 business days after we receive the item.
------------------------------------------------------------

In [52]:
print(answer)

As a ShopEasy Plus member, you have 15 days from delivery to return electronics like headphones. Since the headphones are opened and likely a change-of-mind return, they must be within this 15-day window. You also get free return shipping. To proceed, start a return via "My Orders," select the correct reason, and ShipEasy will process your refund within 3-5 business days after receiving the item.


**A second ticket -- automatic `doc_type` inference.** This time the customer describes a crash and asks "is this a known issue?" The classifier should infer both `product_area=payments` *and* `doc_type=bug_report`, closing the loop with our very first (unfiltered) example.

In [ ]:
user_question = "Customer says the app keeps crashing when they try to pay on their Android phone -- is this a known issue?"

prompt = classify_prompt_template.invoke({"question": user_question})
ticket_filter = structured_llm.invoke(prompt)
print("Classified:", ticket_filter)

pinecone_filter = build_pinecone_filter(ticket_filter)
print("Pinecone filter:", pinecone_filter)

retrieved_docs, answer = generate_filtered_answer(user_question, pinecone_filter)
for doc in retrieved_docs:
    m = doc.metadata
    print(f"[{m['doc_type']} / {m['product_area']}] {m['title']}")
print("\n" + "=" * 80 + "\n")
print(answer)

### Recap

Metadata filtering adds *structure* on top of semantic search -- and it works directly on the vectors we already indexed in Notebook 1:

- **Manual filters** let an agent pinpoint exactly the slice they care about -- e.g. scoping to `doc_type=bug_report` and `product_area=payments` to surface only known checkout bugs. Pinecone also supports operators like `$in`, `$ne`, and `$gt` for richer queries.
- **LLM-classified filters** let the system read a raw customer ticket and auto-scope retrieval to the right `product_area`, `doc_type`, and `customer_tier` -- including the `$in: [tier, "all"]` trick so tier-specific queries don't accidentally drop the general docs that also apply.

The result: instead of 5 loosely-related chunks spanning shipping, payments, and returns, the agent gets back exactly the bug-specific or tier-specific knowledge they need.

Next up, **Notebook 3 -- Hybrid RAG** handles real tickets that mix natural language with exact identifiers (order IDs, SKU codes, tracking numbers) by combining semantic search with keyword (BM25) search.